# 🎭 Face Mask Detection — PyCharm Testing Notebook
### CNN trained on Google Colab → Live Testing with OpenCV
**v2.1 — aligned with `mask_detection_CNN_winning_params.ipynb`**

---

## 📋 Setup Checklist
Copy these files into **the same folder as this notebook**:
- `mask_cnn_final.keras` ← full saved model
- `config.json` ← threshold, class map, input size

> **Breaking change:** the model uses a custom LR schedule (`WarmupCosineSchedule`).
> Cell 3 defines and registers it before loading — **do not skip Cell 3**.

Run cells **top to bottom** before choosing an experiment.


## Cell 1 — Install Dependencies
Run once. Restart kernel after if needed.

In [ ]:
import subprocess, sys
packages = ["tensorflow", "opencv-python", "numpy", "matplotlib", "pillow"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ All packages ready.")


## Cell 2 — Imports & Global Settings


In [ ]:
import os, sys, cv2, json, time, warnings, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow : {tf.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"NumPy      : {np.__version__}")

# ── Visual constants ────────────────────────────────────────────────────
COLOR_MASK    = (0, 200, 0)   # green
COLOR_NO_MASK = (0, 0, 220)   # red
FONT          = cv2.FONT_HERSHEY_SIMPLEX

# ── DNN SSD face detector confidence threshold ───────────────────────
DNN_CONF_THRESHOLD = 0.5

print("\n✅ Imports done. Proceed to Cell 3.")


## Cell 3 — Custom Schedule + Load Model & Config
Defines `WarmupCosineSchedule` (required to deserialise `.keras`),
downloads the OpenCV DNN face-detector weights, and loads the model + config.


In [ ]:
MODEL_PATH      = "mask_cnn_final.keras"   # adjust if file is in a different folder
CONFIG_PATH     = "config.json"
PROTO_PATH      = "deploy.prototxt"
CAFFEMODEL_PATH = "res10_300x300_ssd.caffemodel"

# ── 1. Register WarmupCosineSchedule BEFORE load_model ────────────────────────────
# The training notebook serialises this class into the .keras archive.
# keras.models.load_model raises 'Unknown object: WarmupCosineSchedule'
# without this definition.
import numpy as _np

@tf.keras.utils.register_keras_serializable(package='MaskCNN')
class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup then cosine decay. Exact copy from training notebook v2.1."""
    def __init__(self, initial_lr, warmup_steps, total_steps, alpha=1e-7):
        super().__init__()
        self.initial_lr   = float(initial_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps  = int(total_steps)
        self.alpha        = float(alpha)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup    = tf.cast(self.warmup_steps, tf.float32)
        total     = tf.cast(self.total_steps,  tf.float32)
        warmup_lr = self.initial_lr * (step / (warmup + 1e-8))
        progress  = (step - warmup) / tf.maximum(total - warmup, 1.0)
        cosine_lr = self.initial_lr * (
            self.alpha + (1.0 - self.alpha) * 0.5 *
            (1.0 + tf.cos(_np.pi * progress))
        )
        return tf.where(step < warmup, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'initial_lr'  : self.initial_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps' : self.total_steps,
            'alpha'       : self.alpha,
        }

print("✅ WarmupCosineSchedule defined and registered.")

# ── 2. Download OpenCV DNN face detector ──────────────────────────────────────────
# Mirrors the download block in training notebook Section 14.
_PROTO_URL = (
    "https://raw.githubusercontent.com/opencv/opencv/master/"
    "samples/dnn/face_detector/deploy.prototxt"
)
_MODEL_URL = (
    "https://github.com/opencv/opencv_3rdparty/raw/"
    "dnn_samples_face_detector_20170830/"
    "res10_300x300_ssd_iter_140000.caffemodel"
)

for _url, _path, _name in [
    (_PROTO_URL,  PROTO_PATH,      "deploy.prototxt"),
    (_MODEL_URL,  CAFFEMODEL_PATH, "res10_300x300_ssd.caffemodel"),
]:
    if not os.path.exists(_path):
        print(f"Downloading {_name} ...")
        try:
            urllib.request.urlretrieve(_url, _path)
            print(f"  OK → {_path}  ({os.path.getsize(_path):,} bytes)")
        except Exception as _e:
            print(f"  Download failed: {_e}")
            print("  Get it manually: https://github.com/opencv/opencv_3rdparty")
    else:
        print(f"  Already present: {_path}  ({os.path.getsize(_path):,} bytes)")

FACE_NET = cv2.dnn.readNetFromCaffe(PROTO_PATH, CAFFEMODEL_PATH)
print("✅ OpenCV DNN face detector loaded.")

# ── 3. Load config.json ───────────────────────────────────────────────────────────
FALLBACK = {
    "IMG_SIZE"  : (128, 128),
    "THRESHOLD" : 0.5,
    "CLASS_MAP" : {0: "with_mask", 1: "without_mask"},
}

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as _f:
        _raw_cfg = json.load(_f)
    _infer = _raw_cfg.get("inference", {})
    CFG = {
        "IMG_SIZE" : tuple(_infer.get("input_size",         FALLBACK["IMG_SIZE"])),
        "THRESHOLD": _infer.get("decision_threshold",       FALLBACK["THRESHOLD"]),
        "CLASS_MAP": ({int(k): v for k, v in _infer.get("class_map", {}).items()}
                     or FALLBACK["CLASS_MAP"]),
    }
    print(f"✅ config.json loaded  →  threshold={CFG['THRESHOLD']},  input={CFG['IMG_SIZE']}")
else:
    CFG = FALLBACK.copy()
    print("⚠️  config.json not found — using defaults.")

# ── 4. Load Keras model ───────────────────────────────────────────────────────────
assert os.path.exists(MODEL_PATH), f"❌ '{MODEL_PATH}' not found."
MODEL = keras.models.load_model(
    MODEL_PATH,
    custom_objects={'WarmupCosineSchedule': WarmupCosineSchedule}
)
print(f"✅ Model loaded  |  input shape : {MODEL.input_shape}")
print(f"              |  total params : {MODEL.count_params():,}")
print("✅ All systems ready. Run Cell 4, then pick any experiment.")


## Cell 4 — Core Helper Functions
Run once — all experiments depend on this cell.

**Changes from old version:**
- `detect_faces_dnn()` replaces the Haar cascade with the OpenCV DNN SSD detector
  (same model used in training notebook Section 14).
- `predict_batch()` runs inference on **all faces in one model call** — matches
  `run_detection()` in the training notebook.
- `predict_face()` kept for single-face robustness experiments; uses
  `MODEL(..., training=False)` instead of the slower `MODEL.predict()`.
- `annotate_frame()` now accepts 3-tuples `(label, conf, raw)` and adds the
  red overlay for unmasked faces (mirrors training notebook annotation style).


In [ ]:
def preprocess_face(face_bgr):
    """BGR crop → normalised float32 tensor (1, H, W, 3)."""
    rgb  = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    rsz  = cv2.resize(rgb, CFG["IMG_SIZE"])
    norm = rsz.astype("float32") / 255.0
    return np.expand_dims(norm, axis=0)


def detect_faces_dnn(frame, conf_threshold=DNN_CONF_THRESHOLD):
    """OpenCV DNN SSD face detector → list of (x, y, w, h).

    Handles faces tilted up to ~70° and partially occluded faces.
    Mirrors training notebook Section 14 detect_faces_dnn().
    """
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104, 177, 123))
    FACE_NET.setInput(blob)
    dets  = FACE_NET.forward()
    faces = []
    for i in range(dets.shape[2]):
        if dets[0, 0, i, 2] > conf_threshold:
            box = dets[0, 0, i, 3:7] * np.array([w, h, w, h])
            x1, y1, x2, y2 = box.astype(int)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                faces.append((x1, y1, x2 - x1, y2 - y1))
    return faces


def predict_batch(frame, faces, threshold=None):
    """Batch inference: all faces in ONE model call.

    Returns list of (label, confidence, raw_sigmoid) tuples.
    raw_sigmoid = P(without_mask) because CLASS_MAP[1] = 'without_mask'.
    """
    if not faces:
        return []
    t     = threshold if threshold is not None else CFG["THRESHOLD"]
    batch = np.concatenate(
        [preprocess_face(frame[y:y+fh, x:x+fw]) for (x, y, fw, fh) in faces],
        axis=0
    )
    raws = MODEL(batch, training=False).numpy().ravel()
    out  = []
    for raw in raws:
        raw      = float(raw)
        pred_idx = 1 if raw > t else 0
        label    = CFG["CLASS_MAP"].get(pred_idx, "unknown")
        conf     = raw if pred_idx == 1 else (1.0 - raw)
        out.append((label, conf, raw))
    return out


def predict_face(face_bgr, threshold=None):
    """Single-face helper for robustness experiments (Exp 5, 6, 7).

    Uses MODEL(..., training=False) directly — faster than MODEL.predict().
    Returns (label, confidence, raw_sigmoid).
    """
    t        = threshold if threshold is not None else CFG["THRESHOLD"]
    tensor   = preprocess_face(face_bgr)
    raw      = float(MODEL(tensor, training=False).numpy()[0][0])
    pred_idx = 1 if raw > t else 0
    label    = CFG["CLASS_MAP"].get(pred_idx, "unknown")
    conf     = raw if pred_idx == 1 else (1.0 - raw)
    return label, conf, raw


def annotate_frame(frame, faces, preds):
    """Draw coloured bounding boxes and labels in-place.

    preds: list of (label, confidence, raw_sigmoid) or (label, confidence).
    Adds semi-transparent red overlay for unmasked faces (matches training
    notebook annotate_frame style).
    """
    for (x, y, w, h), pred in zip(faces, preds):
        label, conf = pred[0], pred[1]
        wearing = (label == "with_mask")
        color   = COLOR_MASK if wearing else COLOR_NO_MASK
        display = "MASK" if wearing else "NO MASK"
        text    = f"{display} | {conf*100:.1f}%"
        if not wearing:
            overlay = frame.copy()
            cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 0, 180), -1)
            frame[:] = cv2.addWeighted(overlay, 0.20, frame, 0.80, 0)
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        (tw, th), _ = cv2.getTextSize(text, FONT, 0.55, 2)
        cv2.rectangle(frame, (x, y-th-8), (x+tw+4, y), color, -1)
        cv2.putText(frame, text, (x+2, y-4), FONT, 0.55, (255, 255, 255), 2)
    return frame


print("✅ Helpers ready: detect_faces_dnn | predict_batch | predict_face | annotate_frame")


---
## 🧪 Experiment 1 — Basic Live Webcam Detection
**What it does:** Standard detection with green (mask) and red (no mask) bounding boxes + confidence %.  
**Controls:** `Q` to quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam (index 0)"

while True:
    ret, frame = cap.read()
    if not ret: break

    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)

    annotate_frame(frame, faces, preds)
    cv2.putText(frame, f"Faces: {len(faces)}  |  Q=quit", (10, 25), FONT, 0.6, (220, 220, 220), 1)
    cv2.imshow("EXP 1 — Basic Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 1 done.")


---
## 🧪 Experiment 2 — FPS Counter Overlay
**What it does:** Same as Exp 1 but shows real-time FPS in the top corner.  
**Why useful:** Benchmarks how fast your CPU/GPU handles inference.  
**Controls:** `Q` to quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"
prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret: break

    curr_time = time.time()
    fps = 1.0 / (curr_time - prev_time + 1e-9)
    prev_time = curr_time

    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)

    annotate_frame(frame, faces, preds)
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30), FONT, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, "Q=quit", (10, 60), FONT, 0.55, (180, 180, 180), 1)
    cv2.imshow("EXP 2 — FPS Counter", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 2 done.")


---
## 🧪 Experiment 3 — Confidence Bar Graph (per face)
**What it does:** Draws a mini horizontal bar below each face.  
Green fill = mask probability. Red fill = no-mask probability.  
**Why useful:** Makes the model's uncertainty visually obvious.  
**Controls:** `Q` to quit.


In [ ]:
# Note on sigmoid semantics:
#   raw  = P(without_mask)  [CLASSES[1]]
#   1-raw = P(with_mask)     [CLASSES[0]]
# The green (mask) fill uses (1-raw) and red (no-mask) fill uses raw.

cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"

while True:
    ret, frame = cap.read()
    if not ret: break

    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)

    for (x, y, w, h), (label, conf, raw) in zip(faces, preds):
        color = COLOR_MASK if label == "with_mask" else COLOR_NO_MASK
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame,
                    f"{'MASK' if label == 'with_mask' else 'NO MASK'} {conf*100:.0f}%",
                    (x, y-8), FONT, 0.5, color, 1)

        # Confidence bar: green = P(with_mask) = 1-raw, red = P(without_mask) = raw
        bx, by    = x, y + h + 5
        mask_fill = int(w * (1.0 - raw))    # P(with_mask) fraction
        cv2.rectangle(frame, (bx, by), (bx+w, by+10), (60, 60, 60), -1)
        cv2.rectangle(frame, (bx, by), (bx+mask_fill, by+10), COLOR_MASK, -1)
        cv2.rectangle(frame, (bx+mask_fill, by), (bx+w, by+10), COLOR_NO_MASK, -1)

    cv2.putText(frame, "EXP 3: Conf Bar | Q=quit", (10, 25), FONT, 0.55, (220, 220, 220), 1)
    cv2.imshow("EXP 3 — Confidence Bar", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 3 done.")


---
## 🧪 Experiment 4 — Threshold Slider (Trackbar)
**What it does:** A trackbar lets you adjust the sigmoid decision threshold from 0 to 1 in real time.  
**Why useful:** Directly shows how the boundary affects classification — core concept in binary classifiers.  
**Controls:** Drag slider. `Q` to quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"
win = "EXP 4 — Threshold Slider"
cv2.namedWindow(win)
cv2.createTrackbar("Threshold %", win, int(CFG["THRESHOLD"] * 100), 100, lambda v: None)

while True:
    ret, frame = cap.read()
    if not ret: break

    t_val = cv2.getTrackbarPos("Threshold %", win) / 100.0
    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces, threshold=t_val)

    annotate_frame(frame, faces, preds)
    cv2.putText(frame, f"Threshold: {t_val:.2f} | Q=quit", (10, 25), FONT, 0.6, (0, 255, 255), 1)
    cv2.imshow(win, frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 4 done.")


---
## 🧪 Experiment 5 — Grayscale Input Robustness
**What it does:** Strips color from each face (R=G=B=gray) before sending to the model.  
**Why useful:** Tests if the CNN relies on color or purely on texture/shape. Expect confidence drops!  
**Controls:** `Q` to quit.


In [ ]:
# Face detection runs on the colour frame (DNN SSD needs colour input).
# Only the crop sent to the mask classifier is converted to grayscale.
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"

while True:
    ret, frame = cap.read()
    if not ret: break

    faces     = detect_faces_dnn(frame)
    gray_disp = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_disp = cv2.cvtColor(gray_disp, cv2.COLOR_GRAY2BGR)
    preds     = []

    for (x, y, w, h) in faces:
        face_bgr  = frame[y:y+h, x:x+w]
        g         = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
        gray_face = cv2.merge([g, g, g])               # 3-channel gray
        label, conf, raw = predict_face(gray_face)     # single-face path
        preds.append((label, conf, raw))

    annotate_frame(gray_disp, faces, preds)
    cv2.putText(gray_disp, "EXP 5: Grayscale Input | Q=quit", (10, 25), FONT, 0.55, (200, 200, 200), 1)
    cv2.imshow("EXP 5 — Grayscale Input", gray_disp)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 5 done.")


---
## 🧪 Experiment 6 — Blur Robustness Test
**What it does:** Applies Gaussian blur to each face before inference. Kernel size controlled by trackbar.  
**Why useful:** Simulates low-resolution cameras or people far from the lens. How degraded can input be before the model fails?  
**Controls:** Drag "Blur level" slider. `Q` to quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"
win = "EXP 6 — Blur Test"
cv2.namedWindow(win)
cv2.createTrackbar("Blur level", win, 1, 15, lambda v: None)

while True:
    ret, frame = cap.read()
    if not ret: break

    blur_val = cv2.getTrackbarPos("Blur level", win)
    k        = blur_val * 2 + 1   # must be odd

    # Detect on the unblurred frame so DNN SSD works correctly
    faces = detect_faces_dnn(frame)
    preds = []
    for (x, y, w, h) in faces:
        blurred      = cv2.GaussianBlur(frame[y:y+h, x:x+w], (k, k), 0)
        label, conf, raw = predict_face(blurred)
        preds.append((label, conf, raw))
        frame[y:y+h, x:x+w] = blurred   # show blur on display

    annotate_frame(frame, faces, preds)
    cv2.putText(frame, f"Kernel: {k}x{k} | Q=quit", (10, 25), FONT, 0.6, (0, 255, 255), 1)
    cv2.imshow(win, frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 6 done.")


---
## 🧪 Experiment 7 — Brightness & Contrast Manipulation
**What it does:** Two trackbars to adjust brightness (-100 → +100) and contrast (1x → 3x).  
**Why useful:** Simulates dim rooms, harsh sunlight, or overexposed cameras — real deployment conditions.  
**Controls:** Drag both sliders. `Q` to quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"
win = "EXP 7 — Brightness & Contrast"
cv2.namedWindow(win)
cv2.createTrackbar("Brightness+100", win, 100, 200, lambda v: None)
cv2.createTrackbar("Contrast x10",   win, 10,  30,  lambda v: None)

while True:
    ret, frame = cap.read()
    if not ret: break

    beta     = cv2.getTrackbarPos("Brightness+100", win) - 100
    alpha    = cv2.getTrackbarPos("Contrast x10",   win) / 10.0
    adjusted = cv2.convertScaleAbs(frame, alpha=alpha, beta=beta)

    # Detect on adjusted frame so face detector sees the same image the
    # mask classifier will.
    faces = detect_faces_dnn(adjusted)
    preds = []
    for (x, y, w, h) in faces:
        label, conf, raw = predict_face(adjusted[y:y+h, x:x+w])
        preds.append((label, conf, raw))

    annotate_frame(adjusted, faces, preds)
    cv2.putText(adjusted, f"B={beta:+d}  C={alpha:.1f}x | Q=quit", (10, 25), FONT, 0.6, (0, 255, 255), 1)
    cv2.imshow(win, adjusted)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 7 done.")


---
## 🧪 Experiment 8 — Side-by-Side (Original | Canny Edges)
**What it does:** Splits the view: left = original, right = Canny edge-detected frame. Annotations on both.  
**Why useful:** Shows what low-level visual features (edges) the model is implicitly responding to.  
**Controls:** `Q` to quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"

while True:
    ret, frame = cap.read()
    if not ret: break

    gray     = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges    = cv2.Canny(gray, 50, 150)
    edge_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)

    left  = frame.copy()
    right = edge_bgr.copy()
    annotate_frame(left,  faces, preds)
    annotate_frame(right, faces, preds)
    cv2.putText(left,  "Original",    (10, 25), FONT, 0.65, (220, 220, 220), 1)
    cv2.putText(right, "Canny Edges", (10, 25), FONT, 0.65, (220, 220, 220), 1)

    combined = np.hstack([left, right])
    cv2.putText(combined, "Q=quit", (combined.shape[1]-80, 25), FONT, 0.5, (180, 180, 180), 1)
    cv2.imshow("EXP 8 — Side-by-Side", combined)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 8 done.")


---
## 🧪 Experiment 9 — Session Statistics Logger
**What it does:** Tracks total frames, faces detected, mask/no-mask ratio, and average confidence over the whole session. Prints a full summary when you quit.  
**Controls:** `Q` to quit (summary prints in notebook output).


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"

stats   = {"frames": 0, "faces": 0,
           "with_mask":    {"count": 0, "conf_sum": 0.0},
           "without_mask": {"count": 0, "conf_sum": 0.0}}
t_start = time.time()

while True:
    ret, frame = cap.read()
    if not ret: break

    stats["frames"] += 1
    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)
    stats["faces"] += len(faces)

    for (label, conf, raw) in preds:
        stats[label]["count"]    += 1
        stats[label]["conf_sum"] += conf

    annotate_frame(frame, faces, preds)

    elapsed = time.time() - t_start
    wm  = stats["with_mask"]["count"]
    nm  = stats["without_mask"]["count"]
    tot = wm + nm + 1e-9
    hud = [f"Elapsed: {elapsed:.1f}s", f"Frames: {stats['frames']}",
           f"Faces : {stats['faces']}",
           f"Mask  : {wm} ({wm/tot*100:.1f}%)",
           f"NoMask: {nm} ({nm/tot*100:.1f}%)"]
    for i, line in enumerate(hud):
        cv2.putText(frame, line, (10, 25+i*22), FONT, 0.55, (0, 255, 255), 1)

    cv2.putText(frame, "Q=quit", (10, frame.shape[0]-10), FONT, 0.5, (180, 180, 180), 1)
    cv2.imshow("EXP 9 — Session Stats", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()

wm = stats["with_mask"]; nm = stats["without_mask"]
print("=" * 45)
print("        SESSION SUMMARY")
print("=" * 45)
print(f"  Total frames    : {stats['frames']}")
print(f"  Total detections: {wm['count'] + nm['count']}")
print(f"  With Mask       : {wm['count']}  "
      f"(avg conf {wm['conf_sum']/(wm['count']+1e-9):.2%})")
print(f"  Without Mask    : {nm['count']}  "
      f"(avg conf {nm['conf_sum']/(nm['count']+1e-9):.2%})")
print("=" * 45)


---
## 🧪 Experiment 10 — Video File Testing + Save Annotated Output
**What it does:** Run inference on a local `.mp4` file instead of webcam. Saves the annotated video as `annotated_output.mp4`.  
**Why useful:** Reproducible testing — same video, same results every time.  
**Change** `VIDEO_PATH` to your file path.


In [ ]:
VIDEO_PATH = "test_video.mp4"   # ← Change to your video file
OUT_PATH   = "annotated_output.mp4"

assert os.path.exists(VIDEO_PATH), f"❌ File not found: {VIDEO_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

frame_count = 0
print(f"Processing {VIDEO_PATH} ...")

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_count += 1

    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)

    annotate_frame(frame, faces, preds)
    out.write(frame)
    cv2.imshow("EXP 10 — Video File (Q=quit)", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); out.release(); cv2.destroyAllWindows()
print(f"✅ Done. {frame_count} frames → saved to '{OUT_PATH}'")


---
## 🧪 Experiment 11 — Snapshot + Matplotlib Confidence Chart
**What it does:** Press `S` to freeze a frame. A matplotlib stacked bar chart instantly appears showing the mask vs no-mask probability for every detected face.  
**Why useful:** Great for presentations and understanding model uncertainty on a specific frame.  
**Controls:** `S` = snapshot, `Q` = quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"
print("Press S to take a snapshot chart, Q to quit.")

while True:
    ret, frame = cap.read()
    if not ret: break

    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)

    annotate_frame(frame, faces, preds)
    cv2.putText(frame, "S=snapshot  Q=quit", (10, 25), FONT, 0.6, (0, 255, 255), 1)
    cv2.imshow("EXP 11 — Snapshot", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"): break
    elif key == ord("s") and len(faces) > 0:
        # raw = P(without_mask); 1-raw = P(with_mask)
        raws      = [p[2] for p in preds]
        labels_x  = [f"Face {i+1}" for i in range(len(faces))]
        p_mask    = [1.0 - r for r in raws]
        p_no_mask = raws

        fig, ax = plt.subplots(figsize=(max(5, len(faces)*2), 4))
        x_pos = np.arange(len(labels_x))
        ax.bar(x_pos, p_mask,    color="#2ecc71", label="Mask prob (1−raw)")
        ax.bar(x_pos, p_no_mask, bottom=p_mask,   color="#e74c3c", label="No-Mask prob (raw)")
        ax.axhline(CFG["THRESHOLD"], color="gray", linestyle="--", linewidth=1.5,
                   label=f"Threshold ({CFG['THRESHOLD']})")
        ax.set_xticks(x_pos); ax.set_xticklabels(labels_x)
        ax.set_ylim(0, 1.0); ax.set_ylabel("Probability")
        ax.set_title("Mask vs No-Mask confidence per face")
        ax.legend(); plt.tight_layout()
        plt.savefig("snapshot_chart.png", dpi=120)
        plt.show()
        print("✅ Chart saved → snapshot_chart.png")

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 11 done.")


---
## 🧪 Experiment 12 — No-Mask Alarm Mode
**What it does:** Triggers a flashing red border and "⚠ MASK REQUIRED" warning whenever ANY detected face lacks a mask. Models a real-world access-control system.  
**Controls:** `Q` to quit.


In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "❌ Cannot open webcam"

while True:
    ret, frame = cap.read()
    if not ret: break

    faces = detect_faces_dnn(frame)
    preds = predict_batch(frame, faces)
    alert = any(p[0] == "without_mask" for p in preds)

    annotate_frame(frame, faces, preds)

    if alert:
        if int(time.time() * 2) % 2 == 0:
            cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 12)
        cv2.putText(frame, "MASK REQUIRED", (70, frame.shape[0]//2),
                    FONT, 1.3, (0, 0, 255), 3)
    else:
        cv2.putText(frame, "All Clear", (10, frame.shape[0]-20), FONT, 0.8, (0, 200, 0), 2)

    cv2.putText(frame, "EXP 12: Alarm | Q=quit", (10, 25), FONT, 0.55, (200, 200, 200), 1)
    cv2.imshow("EXP 12 — Alarm Mode", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release(); cv2.destroyAllWindows()
print("✅ Experiment 12 done.")
